In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models
from sklearn.metrics import confusion_matrix
from PIL import Image
import collections

In [21]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(in_features=2048, out_features=3)

In [22]:
#DATA AUGMENTATION

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, hue=0.2, saturation=0.3),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

In [23]:
# DATASETS
train_dataset = torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/KMeans_Dataset_with_ResNet50/train",
                                                 transform=transform_train)

test_dataset = torchvision.datasets.ImageFolder(root="C:/Users/panag/Desktop/Thesis/Datasets/KMeans_Dataset_with_ResNet50/test",
                                                transform=transform_test)

In [24]:
#Train-Test Loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=32,num_workers=2,shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=32,num_workers=2,shuffle=False)

In [25]:
class_counts = collections.Counter(train_dataset.targets)
for cls_idx, count in sorted(class_counts.items()):
    cls_name = train_dataset.classes[cls_idx]
    print(f"  {cls_name}: {count} pictures")

total_samples = len(train_dataset)
class_weights = torch.tensor([
    total_samples / class_counts[i]
    for i in range(len(train_dataset.classes))], dtype=torch.float)

print(f"\nClass weights: {class_weights}")

  High: 803 pictures
  Low: 939 pictures
  NoWaste: 705 pictures

Class weights: tensor([3.0473, 2.6060, 3.4709])


In [26]:
#Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [27]:
#CrossEntropyLoss
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
model = model.to(device)

In [28]:
# FROZEN LAYERS + OPTIMIZER
for param in model.parameters():
    param.requires_grad = False
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

In [29]:
#Optimizer
optimizer = optim.Adam(
    list(model.fc.parameters()) +
    list(model.layer3.parameters()) +
    list(model.layer4.parameters()),
    lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

In [30]:
#TRAINING LOOP
num_epoch = 10
best_accuracy =0.0

for epoch in range(num_epoch):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epoch}, Loss: {avg_loss:.4f}')

    #Evaluation
    model.eval()
    test_loss = 0.0
    correct = 0.0
    total = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs, dim=1)
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_test_loss = test_loss / len(test_loader)
    accuracy = 100 * correct / total
    print(f'Test Loss: {avg_test_loss:.4f}, Test Accuracy: {accuracy:.2f}%\n')

    #Scheduler step
    scheduler.step(avg_test_loss)

    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(),'RN50_WL_TK-Means50.pth')
        print(f'New best model saved! Accuracy: {best_accuracy:.2f}%\n')

    #Confusion Matrix
    cm = confusion_matrix(all_targets, all_predictions)
    print("Confusion Matrix:")
    print(cm)
    print("-" * 40)

Epoch 1/10, Loss: 0.7268
Test Loss: 0.4967, Test Accuracy: 82.03%

New best model saved! Accuracy: 82.03%

Confusion Matrix:
[[144  19  53]
 [  5 211  11]
 [  6  16 147]]
----------------------------------------
Epoch 2/10, Loss: 0.5399
Test Loss: 0.5361, Test Accuracy: 83.33%

New best model saved! Accuracy: 83.33%

Confusion Matrix:
[[192  13  11]
 [ 14 207   6]
 [ 39  19 111]]
----------------------------------------
Epoch 3/10, Loss: 0.4977
Test Loss: 0.8749, Test Accuracy: 79.74%

Confusion Matrix:
[[163   8  45]
 [  4 172  51]
 [ 11   5 153]]
----------------------------------------
Epoch 4/10, Loss: 0.4867
Test Loss: 0.4324, Test Accuracy: 86.11%

New best model saved! Accuracy: 86.11%

Confusion Matrix:
[[199   8   9]
 [ 14 206   7]
 [ 33  14 122]]
----------------------------------------
Epoch 5/10, Loss: 0.4475
Test Loss: 1.3974, Test Accuracy: 66.83%

Confusion Matrix:
[[151   6  59]
 [ 10  99 118]
 [ 10   0 159]]
----------------------------------------
Epoch 6/10, Loss: 0.

In [36]:
def predict_confidence(image_path):
    model.load_state_dict(torch.load('RN50_WL_TK-Means50.pth'))
    model.eval()
    
    image = Image.open(image_path).convert('RGB')
    image = transform_test(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_class = torch.max(probabilities, dim=1)
        
        prob = confidence.item()
        predicted = predicted_class.item()
        raw_confidence = int(prob * 100)

        if predicted == 0:      
            confidence_score = max(85, raw_confidence)    # High  → ≥85%
        elif predicted == 1:   
            confidence_score = max(65, min(84, raw_confidence))  # Low → 65-84%
        else:                   
            confidence_score = min(64, raw_confidence)    # NoWaste → ≤64%
        
        class_names = {0: "High", 1: "Low", 2: "NoWaste"}

        print(f"Κατηγορία: {class_names[predicted]}")
        print(f"Confidence: {confidence_score}%")
        print(f"Raw Softmax: {raw_confidence}%")
        print("---")
        
    return class_names[predicted], confidence_score


# Δοκιμή
class_name, confidence = predict_confidence("C:/Users/panag/Desktop/full.avif")
print(f"Κατηγορία: {class_name}, Confidence: {confidence}%")

Κατηγορία: Low
Confidence: 84%
Raw Softmax: 86%
---
Κατηγορία: Low, Confidence: 84%
